# 관측 가능한 소행성 목록 얻기

* 이 노트북을 구글 코랩에서 실행하고자 한다면 [파일] - [드라이브에 사본 저장]을 하여 본인의 소유로 만든 후에 코드를 실행하거나 수정할 수 있습니다.

* 이 파일은 실제 수업에 사용하므로 필요에 따라 예고 없이 변경될 수 있습니다.

* If you have any questions or comments on this document, please email me(Kiehyun.Park@gmail.com).

* 이 파일(문서)는 공교육 현장에서 수업시간에 자유롭게 사용할 수 있으나, 다른 목적으로 사용할 시에는 사전에 연락을 주셔서 상의해 주시기 바랍니다.

이 자료는 What's observable로 부터 특정한 날 관측할 수 있는 소행성의 목록을 만드는 과정을 설명합니다.

## 필요한 환경

이 프로젝트를 위해서는 아래의 모듈이 필요합니다.

> numpy, pandas, matplotlib, astroquery, requests, astropy, astropy.units, certifi, version_information


### 구글 코랩에 한글 폰트 설치

matplotlib에서 한글을 사용하기 위해서는 한글 폰트가 필요하다. 구글 코랩에서 현재의 Jupyter notebook을 실행한다면 아래 코드를 실행 한 후 런타임 다시 시작을 해 줘야 한글을 사용할 수 있을 것이다.

In [12]:
# !sudo apt-get install -y fonts-nanum
# !sudo fc-cache -fv
# !rm ~/.cache/matplotlib -rf

### 런타임 다시 시작

위의 셀을 실행한 다음 반드시 다음 과정을 잊지 말자.

* [메뉴]-[런타임]-[런터임 다시 시작]

* [메뉴]-[런타임]-[이전 셀 실행]을 해주어야 한다.

### 한글 폰트 사용

위에서 한글 폰트를 설치하고, 런타임 다시시작을 했다면 구글 코랩에서 폰트 경로를 설정하여 한글 사용이 가능해 진다.

In [1]:
#visualization
import matplotlib as mpl
import matplotlib.pylab as plb
import matplotlib.pyplot as plt

# 브라우저에서 바로 그려지도록
%matplotlib inline

# 그래프에 retina display 적용
%config InlineBackend.figure_format = 'retina'

# Colab 의 한글 폰트 설정
plt.rc('font', family='NanumBarunGothic')

# 유니코드에서  음수 부호설정
mpl.rc('axes', unicode_minus=False)

### 모듈 설치 및 버전 확인

아래 셀을 실행하면 이 노트북을 실행하는데 필요한 모듈을 설치하고 파이썬 및 관련 모듈의 버전을 확인할 수 있다.

In [2]:
import importlib, sys, subprocess
packages = "numpy, pandas, matplotlib, astroquery, requests, astropy, astropy.units, certifi, version_information" # required modules
pkgs = packages.split(", ")
for pkg in pkgs :
    if not importlib.util.find_spec(pkg):
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f"**** {pkg} module is now installed.")
    else:
        print(f"******** {pkg} module is already installed.")
%load_ext version_information
import time
now = time.strftime("%Y-%m-%d %H:%M:%S (%Z = GMT%z)")
print(f"This notebook was generated at {now} ")

vv = %version_information {packages}
for i, pkg in enumerate(vv.packages):
    print(f"{i} {pkg[0]:10s} {pkg[1]:s}")

******** numpy module is already installed.
******** pandas module is already installed.
******** matplotlib module is already installed.
******** astroquery module is already installed.
******** requests module is already installed.
******** astropy module is already installed.
******** astropy.units module is already installed.
******** certifi module is already installed.
******** version_information module is already installed.
This notebook was generated at 2023-11-02 10:37:49 (KST = GMT+0900) 
0 Python     3.11.4 64bit [GCC 11.2.0]
1 IPython    8.15.0
2 OS         Linux 5.15.0 82 generic x86_64 with glibc2.31
3 numpy      1.25.2
4 pandas     2.0.3
5 matplotlib 3.7.2
6 astroquery 0.4.6
7 requests   2.31.0
8 astropy    5.1
9 astropy.units The 'astropy.units' distribution was not found and is required by the application
10 certifi    2023.07.22
11 version_information 1.0.4


## 날짜와 시간 다루기

### datetime.datetime

Python 내장 모듈인 datetime을 사용하면 날짜와 시각 객체를 다룰 수 있습니다.

#### 표준시와 세계시

In [3]:
from datetime import datetime, timedelta, timezone

dt_kst_now = datetime.now()
print("현재 시각(KST=GMT+9) :", dt_kst_now)

timezone_utc = timezone(timedelta(hours=-9))
dt_utc_now = dt_kst_now.astimezone(timezone_utc)
print("현재 시각(UTC) :", dt_utc_now)

현재 시각(KST=GMT+9) : 2023-11-02 10:37:55.153918
현재 시각(UTC) : 2023-11-01 16:37:55.153918-09:00


#### fits header의 시간 형식

fits header의 시간 형식은 '2023-09-11T20:19:18'의 형태를 띠고 있는데 datetime.strptime을 이용하면 날짜시간 형식을 바로 datetime 객체로 만들 수 있습니다.

이를 세계시로 바꾸려면 timedelta를 이용할 수 있습니다.

In [4]:
start_dt_kst_str = '2023-09-11T20:19:18'
end_dt_kst_str = '2023-12-31T00:00:00'

start_dt_kst = datetime.strptime(start_dt_kst_str, '%Y-%m-%dT%H:%M:%S')
print(start_dt_kst)
print(type(start_dt_kst))

start_dt_utc = start_dt_kst + timedelta(hours=-9)
print(start_dt_utc)
print(type(start_dt_utc))

2023-09-11 20:19:18
<class 'datetime.datetime'>
2023-09-11 11:19:18
<class 'datetime.datetime'>


#### string 형태로 변환하기

datetime 객체를 string으로 바꿀때는 strftime 메쏘들ㄹ 이용하면 됩니다.

In [5]:

start_dt_utc_str = start_dt_utc.strftime('%Y-%m-%dT%H:%M:%S')
print("start_dt_utc_str :", start_dt_utc_str)
print("type(start_dt_utc_str) :", type(start_dt_utc_str))


start_dt_utc_str : 2023-09-11T11:19:18
type(start_dt_utc_str) : <class 'str'>


#### datetime 이용하여 list 생성

아래와 같이 하면, 시작 시각과 끝나는 시각, 그리고 시간 간격을 정해 주고, Python의 datatime 객체를 이용하여 일정 시간 간격의 목록(list)을 만을 수 있다.

In [6]:
from datetime import datetime, timedelta
start_time_text = "2017-01-01"
end_time_text = "2017-06-01"
dt = 1
def datetime_range(start_dt, end_dt, delta):
    '''
        Parameters
        ----------
        start_dt : datetime.datetime
        end_dt : datetime.datetime
        delta : int
    '''
    current = start_dt
    while current < end_dt:
        yield current
        current += delta

dts = [dt.strftime('%Y-%m-%d') for dt in
            datetime_range(datetime.fromisoformat(start_time_text), datetime.fromisoformat(end_time_text),
            timedelta(days=dt))]

print("len(dts):", len(dts))
print("dts", dts)

len(dts): 151
dts ['2017-01-01', '2017-01-02', '2017-01-03', '2017-01-04', '2017-01-05', '2017-01-06', '2017-01-07', '2017-01-08', '2017-01-09', '2017-01-10', '2017-01-11', '2017-01-12', '2017-01-13', '2017-01-14', '2017-01-15', '2017-01-16', '2017-01-17', '2017-01-18', '2017-01-19', '2017-01-20', '2017-01-21', '2017-01-22', '2017-01-23', '2017-01-24', '2017-01-25', '2017-01-26', '2017-01-27', '2017-01-28', '2017-01-29', '2017-01-30', '2017-01-31', '2017-02-01', '2017-02-02', '2017-02-03', '2017-02-04', '2017-02-05', '2017-02-06', '2017-02-07', '2017-02-08', '2017-02-09', '2017-02-10', '2017-02-11', '2017-02-12', '2017-02-13', '2017-02-14', '2017-02-15', '2017-02-16', '2017-02-17', '2017-02-18', '2017-02-19', '2017-02-20', '2017-02-21', '2017-02-22', '2017-02-23', '2017-02-24', '2017-02-25', '2017-02-26', '2017-02-27', '2017-02-28', '2017-03-01', '2017-03-02', '2017-03-03', '2017-03-04', '2017-03-05', '2017-03-06', '2017-03-07', '2017-03-08', '2017-03-09', '2017-03-10', '2017-03-11', '

### astropy.time

표준시나 세계시 정도는 datetime을 사용해도 무방하지만 천문학에서 사용하는 시간에는 조금 부족합니다. 천문학에서 사용하는 시간계는 datetime 모듈보다는 astropy.time 모듈을 사용하기를 권장합니다.

#### 세계시

In [7]:
from astropy.time import Time

t_kst_now = Time.now()
print("현재 시각(KST=GMT+9) :", t_kst_now)

t_ut_now = Time(datetime.utcnow(), scale='utc', location=('127d', '37d'))
print("현재 시각(UTC) :", t_ut_now)

print("type(t_ut_now) ;", type(t_ut_now))

현재 시각(KST=GMT+9) : 2023-11-02 01:38:05.262271
현재 시각(UTC) : 2023-11-02 01:38:05.263297
type(t_ut_now) ; <class 'astropy.time.core.Time'>


#### dir함수로 메쏘드 알아보기


In [8]:
print("dir(t_ut_now) ;", dir(t_ut_now))

dir(t_ut_now) ; ['FORMATS', 'SCALES', 'T', '_APPLICABLE_FUNCTIONS', '_METHOD_FUNCTIONS', '__abstractmethods__', '__add__', '__array_function__', '__array_priority__', '__bool__', '__class__', '__copy__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getitem__', '__getnewargs__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__radd__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setitem__', '__sizeof__', '__str__', '__sub__', '__subclasshook__', '__weakref__', '_abc_impl', '_advanced_index', '_apply', '_astropy_column_attrs', '_call_erfa', '_format', '_get_delta_tdb_tt', '_get_delta_ut1_utc', '_get_time_fmt', '_init_from_vals', '_make_value_equivalent', '_match_shape', '_set_delta_tdb_tt', '_set_delta_ut1_utc', '_set_scale', '_shaped_like_input', '_sid_time_or_earth_rot_ang', '_

#### Julian day 구하기

In [9]:
print("t_ut_now.to_value(format='jd') ;", t_ut_now.to_value(format='jd'))

t_ut_now.to_value(format='jd') ; 2460250.5681164735


#### 항성시

In [10]:
print("t_ut_now.sidereal_time('mean') :", t_ut_now.sidereal_time('mean'))

t_ut_now.sidereal_time('mean') : 12h50m24.69378127s


#### astropy.time 이용하여 list 생성

일정 시간 간격으로 시뮬레이션 하려면 시간 간격의 리스트를 미리 만들어 두면 편리할 것이다. 다음은 Python의 astropy.time을 이용하여 일정 시간 간격의 목록(list)을 만드는 방법이다.

In [12]:
import numpy as np
from astropy.time import Time, TimeDelta

start_time_text = "2023-11-02"
end_time_text = "2023-11-30"
dt = 1

def astropyTime_range(start_t, end_t, delta):
    '''
        Parameters
        ----------
        start_t : datetime.datetime
        end_t : datetime.datetime
        delta : int
    '''
    current = start_t
    while current < end_t:
        yield current
        current += delta

# dts = [dt.iso for dt in
#        astropyTime_range(Time(start_time_text), Time(end_time_text),
#        TimeDelta(dt, format='jd'))]

# dts = [dt.strftime('%Y-%m-%d') for dt in
#        astropyTime_range(Time(start_time_text), Time(end_time_text),
#        TimeDelta(dt, format='jd'))]

dts = [dt for dt in
        astropyTime_range(Time(start_time_text), Time(end_time_text),
        TimeDelta(dt, format='jd'))]

print(len(dts))
print(dts)

28
[<Time object: scale='utc' format='iso' value=2023-11-02 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-11-03 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-11-04 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-11-05 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-11-06 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-11-07 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-11-08 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-11-09 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-11-10 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-11-11 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-11-12 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-11-13 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-11-14 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-11-15 00:00:00.000>, <T

## What's observable

[What's observable](https://ssd.jpl.nasa.gov/tools/sbwobs.html) 에서는 관측소 코드와 관측 조건들을 설정하여 소행성과 혜성의 관측 대상을 얻을 수 있다.


### Set date & condition

먼저 관측 일자와 조건들을 설정해 보자.

In [13]:
#############################################
# variables
mpc_code='P64' # Observer Location: GSHS Observatory, Suwon [code: P64]

#관측 시각 설정 (현재 시각)
t_ut_now = Time(datetime.utcnow(), scale='utc', location=('127d', '37d'))
print("type(obs_dt_utc) :", type(t_ut_now))
print("현재 시각(UTC) :", t_ut_now)

#관측 시각 설정 (임의의 시각)
# t_ut_now = Time(datetime(2023, 10, 2, 10, 0, 0), scale='utc', location=('127d', '37d'))
# print("type(obs_dt_utc) :", type(t_ut_now))
# print("현재 시각(UTC) :", t_ut_now)

obs_datetime = t_ut_now.strftime('%Y-%m-%d %H:%M:%S')
print("obs_datetime :", obs_datetime)
obs_date = t_ut_now.strftime('%Y%m%d')
print("obs_date :", obs_date)

elev_min=60  #
time_min=15
vmag_min=6
vmag_max=13
list_num=100
sort='trans'

type(obs_dt_utc) : <class 'astropy.time.core.Time'>
현재 시각(UTC) : 2023-11-02 01:39:08.258977
obs_datetime : 2023-11-02 01:39:08
obs_date : 20231102


### api 설정

[Horizons API](https://ssd-api.jpl.nasa.gov/doc/horizons.html)를 제공하고 있어 requests로 받아아올 url을 만들어 보겠습니다.



In [14]:
# Define API URL and SPK filename:
url = 'https://ssd-api.jpl.nasa.gov/sbwobs.api'
spk_filename = 'spk_file.bsp'

# Get the requested SPK-ID from the command-line:
if (len(sys.argv)) == 1:
  print("please specify SPK-ID on the command-line")
  sys.exit(2)
spkid = sys.argv[1]

# Build the appropriate URL for this API request:
# IMPORTANT: You must encode the "=" as "%3D" and the ";" as "%3B" in the
#            Horizons COMMAND parameter specification.
url += f"?sb-kind=a&mpc-code={mpc_code}&obs-time={str(obs_datetime)}&elev-min={elev_min}&time-min={time_min}"
url += f"&vmag-max={vmag_max}&vmag-min={vmag_min}&optical=true&fmt-ra-dec=true&mag-required=true&output-sort={sort}&maxoutput={list_num}"

print("url :", url)

url : https://ssd-api.jpl.nasa.gov/sbwobs.api?sb-kind=a&mpc-code=P64&obs-time=2023-11-02 01:39:08&elev-min=60&time-min=15&vmag-max=13&vmag-min=6&optical=true&fmt-ra-dec=true&mag-required=true&output-sort=trans&maxoutput=100


### reauests.get

api에서 제공하는 내용을 json 형태로 받아 보겠습니다.

In [15]:
import json
import requests
# Submit the API request and decode the JSON-response:
response = requests.get(url)
try:
  data = json.loads(response.text)
  print(data)
except ValueError:
  print("Unable to decode JSON results")

{'signature': {'source': 'NASA/JPL Small-Body Observability API', 'version': '1.0'}, 'obs_constraints': {'mag-required': 'true', 'maxoutput': '100', 'output-sort': 'trans', 'obs-time': '2023-11-02 01:39:08', 'vmag-min': '6', 'fmt-ra-dec': 'true', 'vmag-max': '13', 'elev-min': '60', 'optical': 'true', 'time-min': '15', 'mpc-code': 'P64'}, 'sb_constraints': {'sb-kind': 'a'}, 'location': 'GSHS Observatory, Suwon', 'obs_night': {'sun_set_az': '252.6', 'sun_set': '2023-11-01 08:33', 'end_civil': '2023-11-01 09:00', 'end_nautical': '2023-11-01 09:31', 'end_astronomical': '2023-11-01 10:02', 'begin_astronomical': '2023-11-01 20:27', 'begin_nautical': '2023-11-01 20:57', 'begin_civil': '2023-11-01 21:28', 'sun_rise_az': '107.6', 'sun_rise': '2023-11-01 21:55', 'transit_phase': '0.83', 'transit': '2023-11-01 18:40 ', 'moon_set_phase': '0.80', 'moon_set': '2023-11-02 02:34 ', 'moon_rise_phase': '0.85', 'moon_rise': '2023-11-01 10:48 ', 'begin_dark': '2023-11-01 10:02', 'mid_dark': '2023-11-01 15

안에 들어 있는 데이터를 확인해 보곘습니다.

In [16]:
print("type(data) :", type(data))
print("data.keys :", data.keys)

type(data) : <class 'dict'>
data.keys : <built-in method keys of dict object at 0x7f5d70280980>


 아마도 data['fields']에 있는 것이 필요한 자료인것 같습니다.

In [17]:
print(data['obs_constraints'])
print(data['sb_constraints'])
print(data['fields'])
print(data['data'])
print(type(data['fields']))
print(len(data['fields']))
print(len(data['data'][0]))


{'mag-required': 'true', 'maxoutput': '100', 'output-sort': 'trans', 'obs-time': '2023-11-02 01:39:08', 'vmag-min': '6', 'fmt-ra-dec': 'true', 'vmag-max': '13', 'elev-min': '60', 'optical': 'true', 'time-min': '15', 'mpc-code': 'P64'}
{'sb-kind': 'a'}
['Designation', 'Full name', 'Rise time', 'Transit time', 'Set time', 'Max. time observable', 'R.A.', 'Dec.', 'Vmag', 'Helio. range (au)', 'Topo.range (au)', 'Object-Observer-Sun (deg)', 'Object-Observer-Moon (deg)', 'Galactic latitude (deg)']
[['602', '602 Marianna (A906 DJ)', '10:52', '12:10', '13:28', '02:36', '23:20:33', '+12 58\'08"', '12.3', '2.31', '1.50', '134.96', '89.48', '-44.21'], ['678', '678 Fredegundis (A909 BQ)', '11:39', '12:44', '13:49', '02:09', '23:54:49', '+11 06\'17"', '12.2', '2.03', '1.15', '142.97', '82.92', '-49.40'], ['120', '120 Lachesis (A872 GB)', '12:27', '13:52', '15:18', '02:51', '01:03:20', '+14 10\'01"', '12.8', '3.28', '2.33', '159.93', '66.54', '-48.75'], ['639', '639 Latona (A907 OA)', '12:09', '14:07

### dataframe 만들기

아무래도 가장 다루기 쉬운 데이타 형태인 dataframe으로 만들어 보겠습니다.

In [18]:
import pandas as pd

df = pd.DataFrame.from_dict(data['data'])
df = df.set_axis((data['fields']), axis=1)
df['OBSdate(UT)'] = obs_date
df

,Designation,Full name,Rise time,Transit time,Set time,Max. time observable,R.A.,Dec.,Vmag,Helio. range (au),Topo.range (au),Object-Observer-Sun (deg),Object-Observer-Moon (deg),Galactic latitude (deg),OBSdate(UT)
0,602,602 Marianna (A906 DJ),10:52,12:10,13:28,02:36,23:20:33,"+12 58'08""",12.3,2.31,1.50,134.96,89.48,-44.21,20231102
1,678,678 Fredegundis (A909 BQ),11:39,12:44,13:49,02:09,23:54:49,"+11 06'17""",12.2,2.03,1.15,142.97,82.92,-49.40,20231102
2,120,120 Lachesis (A872 GB),12:27,13:52,15:18,02:51,01:03:20,"+14 10'01""",12.8,3.28,2.33,159.93,66.54,-48.75,20231102
3,639,639 Latona (A907 OA),12:09,14:07,16:05,03:56,01:18:08,"+21 36'40""",12.3,2.75,1.79,162.33,60.43,-41.00,20231102
4,123,123 Brunhild (A872 OB),12:24,14:20,16:16,03:51,01:30:31,"+20 58'14""",12.3,2.43,1.46,165.24,57.95,-41.16,20231102
5,135,135 Hertha (A874 DA),13:04,14:24,15:45,02:41,01:35:23,"+13 20'30""",10.9,2.16,1.18,167.61,59.79,-48.35,20231102
6,122,122 Gerda (A872 OA),13:44,14:36,15:27,01:43,01:46:57,"+09 39'54""",12.4,3.25,2.27,169.26,58.83,-51.04,20231102
7,509,509 Iolanda (A903 HD),13:37,15:01,16:25,02:47,02:12:06,"+13 48'19""",12.1,2.78,1.79,176.51,51.48,-44.87,20231102
8,127,127 Johanna (A872 VB),13:34,15:04,16:34,03:00,02:14:59,"+14 58'25""",12.1,2.74,1.74,177.24,50.37,-43.51,20231102
9,343,343 Ostara (A892 VA),13:44,15:06,16:28,02:44,02:16:53,"+13 33'46""",13.0,1.86,0.871,177.56,50.53,-44.57,20231102


### 관측 계획 저장 폴더 생성

FITS 파일을 저장할 폴더를 "OBSplan_NEA" 이라는 이름으로 생성해보자.

* 만약 리눅스 시스템 이라면 shell 명령어로 가능한데, "!"를 붙이면 shell 명령어를 실행할 수 있다.
> !mkdir OBSplan_NEA

OS의 영향을 받지 않기 위하여 pathlib을 사용하여 폴더를 생성해 보자.

In [19]:
import os
from pathlib import Path
BASEPATH = Path("./")
save_dir_name = "OBSplan_NEA"
print(f"BASEPATH: {BASEPATH}")

if not (BASEPATH/save_dir_name).exists():
    os.mkdir(str(BASEPATH/save_dir_name))
    print (f"{str(BASEPATH/save_dir_name)} is created...")
else :
    print (f"{str(BASEPATH/save_dir_name)} is already exist...")

if not (BASEPATH/save_dir_name/obs_date).exists():
    os.mkdir(str(BASEPATH/save_dir_name/obs_date))
    print (f"{str(BASEPATH/save_dir_name/obs_date)} is created...")
else :
    print (f"{str(BASEPATH/save_dir_name/obs_date)} is already exist...")

BASEPATH: .
OBSplan_NEA is already exist...
OBSplan_NEA/20231102 is already exist...


### dataframe 저장

In [20]:
df.to_csv(str(BASEPATH/save_dir_name/ f'OBSPlan_NEA_{obs_date}.csv'))

##(과제)

12월 1달 동안의 NEA 목록을 csv  파일로 저장하는 코드를 완성하세요.



In [21]:
from datetime import datetime, timedelta
start_dt = datetime.strptime("2023-11-02 10:00:00", '%Y-%m-%d %H:%M:%S')
end_dt = datetime.strptime("2023-11-30 10:00:00", '%Y-%m-%d %H:%M:%S')
dt = 1

def datetime_range(start_dt, end_dt, delta):
    '''
        Parameters
        ----------
        start_dt : datetime.datetime
        end_dt : datetime.datetime
        delta : int
    '''
    current = start_dt
    while current < end_dt:
        yield current
        current += delta

dts = [dt for dt in
            datetime_range(start_dt, end_dt,
            timedelta(days=dt))]

print("len(dts):", len(dts))
print("dts", dts)

len(dts): 28
dts [datetime.datetime(2023, 11, 2, 10, 0), datetime.datetime(2023, 11, 3, 10, 0), datetime.datetime(2023, 11, 4, 10, 0), datetime.datetime(2023, 11, 5, 10, 0), datetime.datetime(2023, 11, 6, 10, 0), datetime.datetime(2023, 11, 7, 10, 0), datetime.datetime(2023, 11, 8, 10, 0), datetime.datetime(2023, 11, 9, 10, 0), datetime.datetime(2023, 11, 10, 10, 0), datetime.datetime(2023, 11, 11, 10, 0), datetime.datetime(2023, 11, 12, 10, 0), datetime.datetime(2023, 11, 13, 10, 0), datetime.datetime(2023, 11, 14, 10, 0), datetime.datetime(2023, 11, 15, 10, 0), datetime.datetime(2023, 11, 16, 10, 0), datetime.datetime(2023, 11, 17, 10, 0), datetime.datetime(2023, 11, 18, 10, 0), datetime.datetime(2023, 11, 19, 10, 0), datetime.datetime(2023, 11, 20, 10, 0), datetime.datetime(2023, 11, 21, 10, 0), datetime.datetime(2023, 11, 22, 10, 0), datetime.datetime(2023, 11, 23, 10, 0), datetime.datetime(2023, 11, 24, 10, 0), datetime.datetime(2023, 11, 25, 10, 0), datetime.datetime(2023, 11, 26

In [22]:
#(과제) 아래 코딩을 완성하여 제출하시오.

#############################################
# variables
mpc_code='P64' # Observer Location: GSHS Observatory, Suwon [code: P64]

elev_min=60 #
time_min=15
vmag_min=6
vmag_max=13
list_num=50
sort='trans'

print(len(dts))
print(dts)
import pandas as pd

df_all = pd.DataFrame()

for dt in dts :
    obs_datetime = dt.strftime('%Y-%m-%d %H:%M:%S')
    print("obs_datetime :", obs_datetime)
    obs_date = dt.strftime('%Y%m%d')
    print("obs_date :", obs_date)

    # Define API URL and SPK filename:
    url = 'https://ssd-api.jpl.nasa.gov/sbwobs.api'
    spk_filename = 'spk_file.bsp'

    # Get the requested SPK-ID from the command-line:
    if (len(sys.argv)) == 1:
        print("please specify SPK-ID on the command-line")
        sys.exit(2)
    spkid = sys.argv[1]

    # Build the appropriate URL for this API request:
    # IMPORTANT: You must encode the "=" as "%3D" and the ";" as "%3B" in the
    #            Horizons COMMAND parameter specification.
    url += f"?sb-kind=a&mpc-code={mpc_code}&obs-time={str(obs_datetime)}&elev-min={elev_min}&time-min={time_min}"
    url += f"&vmag-max={vmag_max}&vmag-min={vmag_min}&optical=true&fmt-ra-dec=true&mag-required=true&output-sort={sort}&maxoutput={list_num}"

    print("url :", url)

    import json
    import requests
    # Submit the API request and decode the JSON-response:
    response = requests.get(url)
    try:
        data = json.loads(response.text)
        print(data)
    except ValueError:
        print("Unable to decode JSON results")

    df = pd.DataFrame.from_dict(data['data'])
    df = df.set_axis((data['fields']), axis=1)
    df['OBSdate(UT)'] = obs_date
    #print(df)

    df.to_csv(str(BASEPATH/save_dir_name/f"OBSPlan_NEA_{dt.strftime('%Y%m%d')}.csv"))

    df_all = pd.concat([df_all, df], axis = 0)

df_all.reset_index(inplace=True)
print("df_all :", df_all)
df_all.to_csv(str(BASEPATH/save_dir_name/f"OBSPlan_NEA_all_{dts[0].strftime('%Y%m%d')}-{dts[-1].strftime('%Y%m%d')}.csv"))


28
[datetime.datetime(2023, 11, 2, 10, 0), datetime.datetime(2023, 11, 3, 10, 0), datetime.datetime(2023, 11, 4, 10, 0), datetime.datetime(2023, 11, 5, 10, 0), datetime.datetime(2023, 11, 6, 10, 0), datetime.datetime(2023, 11, 7, 10, 0), datetime.datetime(2023, 11, 8, 10, 0), datetime.datetime(2023, 11, 9, 10, 0), datetime.datetime(2023, 11, 10, 10, 0), datetime.datetime(2023, 11, 11, 10, 0), datetime.datetime(2023, 11, 12, 10, 0), datetime.datetime(2023, 11, 13, 10, 0), datetime.datetime(2023, 11, 14, 10, 0), datetime.datetime(2023, 11, 15, 10, 0), datetime.datetime(2023, 11, 16, 10, 0), datetime.datetime(2023, 11, 17, 10, 0), datetime.datetime(2023, 11, 18, 10, 0), datetime.datetime(2023, 11, 19, 10, 0), datetime.datetime(2023, 11, 20, 10, 0), datetime.datetime(2023, 11, 21, 10, 0), datetime.datetime(2023, 11, 22, 10, 0), datetime.datetime(2023, 11, 23, 10, 0), datetime.datetime(2023, 11, 24, 10, 0), datetime.datetime(2023, 11, 25, 10, 0), datetime.datetime(2023, 11, 26, 10, 0), date




저장

In [ ]:
df_all.to_csv(str(BASEPATH/save_dir_name/f"OBSPlan_NEA_all_{dts[0].strftime('%Y%m%d')}-{dts[-1].strftime('%Y%m%d')}.csv"))